In [0]:
# COMMAND ----------
# DBTITLE 1,Importações e Configuração Inicial
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit

spark = SparkSession.builder.getOrCreate()

# Definição de Variáveis e Ambientes
SECRET_SCOPE = "salesforce_credentials"
CATALOGO = "bradesco_prod"
SCHEMA = "crm_bronze"
TABELA_BRONZE = f"{CATALOGO}.{SCHEMA}.opportunity_raw"

# COMMAND ----------
# DBTITLE 2,Leitura de Credenciais Seguras
# Busca as credenciais cadastradas no Secret Scope
SF_CLIENT_ID = dbutils.secrets.get(SECRET_SCOPE, "client_id")
SF_CLIENT_SECRET = dbutils.secrets.get(SECRET_SCOPE, "client_secret")

# COMMAND ----------
# DBTITLE 3,Identificação do Watermark (Carga Incremental)
if spark.catalog.tableExists(TABELA_BRONZE):
    watermark_val = spark.sql(f"SELECT MAX(SystemModstamp) FROM {TABELA_BRONZE}").collect()[0][0]
    watermark = watermark_val if watermark_val else "2020-01-01T00:00:00Z"
else:
    watermark = "2020-01-01T00:00:00Z"

print(f"Data limite para extracao (Watermark): {watermark}")

# COMMAND ----------
# DBTITLE 4,Extração de Dados do Salesforce
query_sf = f"""
    SELECT Id, Name, Amount, StageName, AccountId, SystemModstamp
    FROM Opportunity
    WHERE SystemModstamp > {watermark}
"""

df_sf = (
    spark.read.format("salesforce")
    .option("query", query_sf)
    .option("clientId", SF_CLIENT_ID)
    .option("clientSecret", SF_CLIENT_SECRET)
    .load()
)

# COMMAND ----------
# DBTITLE 5,Adição de Metadados e Gravação na Camada Bronze
df_bronze = (
    df_sf
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source", lit("salesforce_bulk_api"))
)

(
    df_bronze.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(TABELA_BRONZE)
)

print(f"Carga Bronze concluida com sucesso na tabela: {TABELA_BRONZE}")